In [1]:
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from scipy.fft import fft, ifft, fftfreq
from scipy.linalg import svd
from scipy.stats import wilcoxon
from statsmodels.tsa.seasonal import STL
import warnings
warnings.filterwarnings('ignore')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")

# -------------------------------------------------------
# Load models
# -------------------------------------------------------
import sys
sys.path.insert(0, './panda')

from panda.patchtst.pipeline import PatchTSTPipeline
from chronos import ChronosPipeline

panda_model = PatchTSTPipeline.from_pretrained(
    mode="predict",
    pretrain_path="GilpinLab/panda",
    device_map=device,
)

chronos_model = ChronosPipeline.from_pretrained(
    "amazon/chronos-t5-small",
    device_map=device,
    torch_dtype=torch.bfloat16,
)

print("Models loaded.")

Device: cpu
Models loaded.


In [4]:
# -------------------------------------------------------
# Metrics
# -------------------------------------------------------
def mae(y_true, y_pred):
    return float(np.mean(np.abs(y_true - y_pred)))

def mse(y_true, y_pred):
    return float(np.mean((y_true - y_pred)**2))


# -------------------------------------------------------
# Per-window normalisation
# -------------------------------------------------------
def instance_norm_window(x_CT):
    """x_CT: (C, T). Normalise per channel using this window only."""
    mu  = x_CT.mean(axis=1, keepdims=True)
    std = x_CT.std( axis=1, keepdims=True) + 1e-8
    return (x_CT - mu) / std, mu, std


def load_ts(path):
    """Raw (C, T) — no global normalisation."""
    df  = pd.read_csv(path)
    df  = df.select_dtypes(include=[np.number])
    return df.values.astype(np.float32).T  # (C, T)


# -------------------------------------------------------
# Inference
# -------------------------------------------------------
CONTEXT_LEN = 512


def panda_forecast(context_np, horizon):
    """context_np: (C, T) normalised. Returns (C, horizon)."""
    TRAIN_H   = 128
    remaining = horizon
    ctx       = context_np.copy()
    preds     = []

    while remaining > 0:
        h         = min(TRAIN_H, remaining)
        context_t = torch.tensor(ctx.T, dtype=torch.float32)
        with torch.no_grad():
            pred = panda_model.predict(
                context_t, h,
                limit_prediction_length=False,
                sliding_context=True,
            )
        p = pred.squeeze().cpu().numpy()
        if p.ndim == 1:
            p = p[:, None]
        if p.shape[0] != context_np.shape[0]:
            p = p.T
        preds.append(p[:, :h])
        ctx       = np.concatenate([ctx[:, h:], p[:, :h]], axis=1)
        remaining -= h

    return np.concatenate(preds, axis=1)  # (C, horizon)


def panda_forecast_univariate(context_np, horizon):
    """
    Panda treating each channel independently.
    Suppresses channel attention.
    Handles horizon > 128 via autoregression.
    context_np: (C, T). Returns (C, horizon).
    """
    TRAIN_H = 128
    C       = context_np.shape[0]
    preds   = []

    for c in range(C):
        remaining = horizon
        ctx_c     = context_np[c:c+1, :].copy()  # (1, T)
        ch_preds  = []

        while remaining > 0:
            h     = min(TRAIN_H, remaining)
            ctx_t = torch.tensor(ctx_c.T, dtype=torch.float32)
            with torch.no_grad():
                pred = panda_model.predict(
                    ctx_t, h,
                    limit_prediction_length=False,
                    sliding_context=True,
                )
            p = pred.squeeze().cpu().numpy()
            if p.ndim == 0:
                p = np.array([float(p)])
            p = p[:h]
            ch_preds.append(p)
            # Roll context forward
            ctx_c     = np.concatenate(
                [ctx_c[:, h:], p[None, :]], axis=1
            )
            remaining -= h

        preds.append(np.concatenate(ch_preds))  # (horizon,)

    return np.stack(preds, axis=0)  # (C, horizon)


def chronos_forecast(context_np, horizon):
    """Batched — all channels in one call."""
    ctx = torch.tensor(context_np, dtype=torch.float32)
    with torch.no_grad():
        out = chronos_model.predict(
            ctx, prediction_length=horizon, num_samples=1
        )
    return out[:, 0, :].cpu().numpy()  # (C, horizon)


# -------------------------------------------------------
# Core evaluator
# -------------------------------------------------------
def evaluate(data_CT, horizon, n_windows=8, label="",
             fn_a=None, fn_b=None,
             name_a="panda", name_b="chronos"):
    """
    data_CT: (C, T) RAW.
    Normalises each window independently.
    fn_a, fn_b: forecast functions (context_normed, horizon) -> (C, H)
    """
    if fn_a is None: fn_a = panda_forecast
    if fn_b is None: fn_b = chronos_forecast

    C, T      = data_CT.shape
    max_start = T - CONTEXT_LEN - horizon
    if max_start <= 0:
        print(f"  [SKIP] {label}: T={T} too short")
        return None

    starts = np.linspace(0, max_start, n_windows, dtype=int)
    mae_a, mae_b = [], []

    for s in starts:
        ctx_raw          = data_CT[:, s : s + CONTEXT_LEN]
        tgt_raw          = data_CT[:, s + CONTEXT_LEN
                                      : s + CONTEXT_LEN + horizon]
        ctx_norm, mu, std = instance_norm_window(ctx_raw)
        tgt_norm          = (tgt_raw - mu) / std

        mae_a.append(mae(tgt_norm, fn_a(ctx_norm, horizon)))
        mae_b.append(mae(tgt_norm, fn_b(ctx_norm, horizon)))

    diff = np.array(mae_b) - np.array(mae_a)
    try:
        _, pval = wilcoxon(diff, alternative="greater") \
            if np.any(diff != 0) else (0, 1.0)
    except Exception:
        pval = np.nan

    adv = np.median(mae_b) - np.median(mae_a)
    sig = " *" if pval < 0.05 else (" ~" if pval < 0.10 else "")

    result = {
        "label"         : label,
        "horizon"       : horizon,
        "name_a"        : name_a,
        "name_b"        : name_b,
        f"{name_a}_mae" : np.median(mae_a),
        f"{name_a}_iqr" : np.percentile(mae_a,75)-np.percentile(mae_a,25),
        f"{name_b}_mae" : np.median(mae_b),
        f"{name_b}_iqr" : np.percentile(mae_b,75)-np.percentile(mae_b,25),
        "advantage_mae" : adv,
        "wilcoxon_p"    : pval,
    }

    print(
        f"  {label:48s}  H={horizon:4d}  "
        f"{name_a}={np.median(mae_a):.4f}[±{result[f'{name_a}_iqr']:.4f}]  "
        f"{name_b}={np.median(mae_b):.4f}[±{result[f'{name_b}_iqr']:.4f}]  "
        f"Adv={adv:+.4f}  p={pval:.3f}{sig}"
    )
    return result


print("Helpers defined.")

Helpers defined.


In [ ]:
DATA_DIR = "./ts_data"   # adjust if needed

dataset_periods = {
    "ETTh1"  : 24,
    "ETTh2"  : 24,
    "Weather": 144,
}

datasets = {
    "ETTh1"  : f"{DATA_DIR}/ETTh1.csv",
    "ETTh2"  : f"{DATA_DIR}/ETTh2.csv",
    "Weather": f"{DATA_DIR}/weather.csv",
}

HORIZONS = [96, 192, 336, 720]

print("Fixed Exp 2.1: Standard Horizon Evaluation")
print("Per-window normalisation, n_windows=20, Wilcoxon tests")
print("-" * 75)

exp21_results = []

for dname, dpath in datasets.items():
    data = load_ts(dpath)
    print(f"\n  {dname}: shape {data.shape}")
    for h in HORIZONS:
        res = evaluate(data, h, n_windows=20,
                       label=f"{dname}_H{h}")
        if res:
            res["dataset"] = dname
            exp21_results.append(res)

df_21 = pd.DataFrame(exp21_results)
df_21.to_csv("fixed_exp21_results.csv", index=False)
print("\nSaved fixed_exp21_results.csv")

Fixed Exp 2.1: Standard Horizon Evaluation
Per-window normalisation, n_windows=20, Wilcoxon tests
---------------------------------------------------------------------------

  ETTh1: shape (7, 17420)


We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of l

  ETTh1_H96                                      H=  96  panda=0.7269[±0.2203]  chronos=0.6633[±0.2388]  Adv=-0.0636  p=0.844


We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of l

  ETTh1_H192                                     H= 192  panda=0.8185[±0.3046]  chronos=0.7825[±0.2742]  Adv=-0.0360  p=0.826


...

We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of l

  ETTh2_H720                                     H= 720  panda=1.1139[±0.2194]  chronos=1.1027[±0.3862]  Adv=-0.0112  p=0.392

  Weather: shape (21, 52696)


We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of l

  Weather_H96                                    H=  96  panda=0.6378[±0.1723]  chronos=0.8115[±0.2036]  Adv=+0.1737  p=0.000 *


We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 


In [1]:
# Reconstruct from terminal output — do not rerun these
exp21_results = [
    # ETTh1
    {"label": "ETTh1_H96",  "horizon": 96,  "dataset": "ETTh1",
     "panda_mae": 0.7269, "panda_mae_iqr": 0.2203,
     "chronos_mae": 0.6633, "chronos_mae_iqr": 0.2388,
     "advantage_mae": -0.0636, "wilcoxon_p": 0.844},
    {"label": "ETTh1_H192", "horizon": 192, "dataset": "ETTh1",
     "panda_mae": 0.8185, "panda_mae_iqr": 0.3046,
     "chronos_mae": 0.7825, "chronos_mae_iqr": 0.2742,
     "advantage_mae": -0.0360, "wilcoxon_p": 0.826},
    {"label": "ETTh1_H336", "horizon": 336, "dataset": "ETTh1",
     "panda_mae": 0.8571, "panda_mae_iqr": 0.1971,
     "chronos_mae": 0.9013, "chronos_mae_iqr": 0.1859,
     "advantage_mae": +0.0441, "wilcoxon_p": 0.649},
    {"label": "ETTh1_H720", "horizon": 720, "dataset": "ETTh1",
     "panda_mae": 0.9921, "panda_mae_iqr": 0.3226,
     "chronos_mae": 1.0189, "chronos_mae_iqr": 0.3845,
     "advantage_mae": +0.0267, "wilcoxon_p": 0.774},
    # ETTh2
    {"label": "ETTh2_H96",  "horizon": 96,  "dataset": "ETTh2",
     "panda_mae": 0.8736, "panda_mae_iqr": 0.4790,
     "chronos_mae": 0.9494, "chronos_mae_iqr": 0.5583,
     "advantage_mae": +0.0758, "wilcoxon_p": 0.478},
    {"label": "ETTh2_H192", "horizon": 192, "dataset": "ETTh2",
     "panda_mae": 0.9697, "panda_mae_iqr": 0.5380,
     "chronos_mae": 0.9505, "chronos_mae_iqr": 0.5730,
     "advantage_mae": -0.0191, "wilcoxon_p": 0.147},
    {"label": "ETTh2_H336", "horizon": 336, "dataset": "ETTh2",
     "panda_mae": 0.9255, "panda_mae_iqr": 0.3860,
     "chronos_mae": 1.1101, "chronos_mae_iqr": 0.3831,
     "advantage_mae": +0.1845, "wilcoxon_p": 0.013},
    {"label": "ETTh2_H720", "horizon": 720, "dataset": "ETTh2",
     "panda_mae": 1.1139, "panda_mae_iqr": 0.2194,
     "chronos_mae": 1.1027, "chronos_mae_iqr": 0.3862,
     "advantage_mae": -0.0112, "wilcoxon_p": 0.392},
    # Weather H96 only
    {"label": "Weather_H96", "horizon": 96, "dataset": "Weather",
     "panda_mae": 0.6378, "panda_mae_iqr": 0.1723,
     "chronos_mae": 0.8115, "chronos_mae_iqr": 0.2036,
     "advantage_mae": +0.1737, "wilcoxon_p": 0.000},
]

print(f"Reconstructed {len(exp21_results)} completed rows.")

Reconstructed 9 completed rows.


In [ ]:
# Resume Exp 2.1 from Weather H=192
DATA_DIR = "./ts_data"

dataset_periods = {
    "ETTh1"  : 24,
    "ETTh2"  : 24,
    "Weather": 144,
}

datasets = {
    "Weather": f"{DATA_DIR}/weather.csv",
    # ETTh1 and ETTh2 already done — skip them
}

HORIZONS_REMAINING = [192, 336, 720]
# H=96 for Weather already done — skip it

exp21_resume = []

data = load_ts(f"{DATA_DIR}/weather.csv")
print(f"Weather: shape {data.shape}")

for h in HORIZONS_REMAINING:
    res = evaluate(data, h, n_windows=20,
                   label=f"Weather_H{h}")
    if res:
        res["dataset"] = "Weather"
        exp21_resume.append(res)

df_21 = pd.DataFrame(exp21_resume)
df_21.to_csv("fixed_exp21_weather_resume.csv", index=False)
print("Saved fixed_exp21_weather_resume.csv")

Weather: shape (21, 52696)


We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of l

  Weather_H192                                   H= 192  panda=0.7224[±0.2385]  chronos=0.9582[±0.2089]  Adv=+0.2358  p=0.001 *


We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of l

  Weather_H336                                   H= 336  panda=0.8481[±0.2898]  chronos=1.0843[±0.3507]  Adv=+0.2362  p=0.000 *


We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 


In [1]:
# Reconstruct from terminal output — do not rerun these
exp21_results = [
    # ETTh1
    {"label": "ETTh1_H96",  "horizon": 96,  "dataset": "ETTh1",
     "panda_mae": 0.7269, "panda_mae_iqr": 0.2203,
     "chronos_mae": 0.6633, "chronos_mae_iqr": 0.2388,
     "advantage_mae": -0.0636, "wilcoxon_p": 0.844},
    {"label": "ETTh1_H192", "horizon": 192, "dataset": "ETTh1",
     "panda_mae": 0.8185, "panda_mae_iqr": 0.3046,
     "chronos_mae": 0.7825, "chronos_mae_iqr": 0.2742,
     "advantage_mae": -0.0360, "wilcoxon_p": 0.826},
    {"label": "ETTh1_H336", "horizon": 336, "dataset": "ETTh1",
     "panda_mae": 0.8571, "panda_mae_iqr": 0.1971,
     "chronos_mae": 0.9013, "chronos_mae_iqr": 0.1859,
     "advantage_mae": +0.0441, "wilcoxon_p": 0.649},
    {"label": "ETTh1_H720", "horizon": 720, "dataset": "ETTh1",
     "panda_mae": 0.9921, "panda_mae_iqr": 0.3226,
     "chronos_mae": 1.0189, "chronos_mae_iqr": 0.3845,
     "advantage_mae": +0.0267, "wilcoxon_p": 0.774},
    # ETTh2
    {"label": "ETTh2_H96",  "horizon": 96,  "dataset": "ETTh2",
     "panda_mae": 0.8736, "panda_mae_iqr": 0.4790,
     "chronos_mae": 0.9494, "chronos_mae_iqr": 0.5583,
     "advantage_mae": +0.0758, "wilcoxon_p": 0.478},
    {"label": "ETTh2_H192", "horizon": 192, "dataset": "ETTh2",
     "panda_mae": 0.9697, "panda_mae_iqr": 0.5380,
     "chronos_mae": 0.9505, "chronos_mae_iqr": 0.5730,
     "advantage_mae": -0.0191, "wilcoxon_p": 0.147},
    {"label": "ETTh2_H336", "horizon": 336, "dataset": "ETTh2",
     "panda_mae": 0.9255, "panda_mae_iqr": 0.3860,
     "chronos_mae": 1.1101, "chronos_mae_iqr": 0.3831,
     "advantage_mae": +0.1845, "wilcoxon_p": 0.013},
    {"label": "ETTh2_H720", "horizon": 720, "dataset": "ETTh2",
     "panda_mae": 1.1139, "panda_mae_iqr": 0.2194,
     "chronos_mae": 1.1027, "chronos_mae_iqr": 0.3862,
     "advantage_mae": -0.0112, "wilcoxon_p": 0.392},
    # Weather H96 only
    {"label": "Weather_H96", "horizon": 96, "dataset": "Weather",
     "panda_mae": 0.6378, "panda_mae_iqr": 0.1723,
     "chronos_mae": 0.8115, "chronos_mae_iqr": 0.2036,
     "advantage_mae": +0.1737, "wilcoxon_p": 0.000},

        # Add these to exp21_results list in Cell 1
    {"label": "Weather_H192", "horizon": 192, "dataset": "Weather",
     "panda_mae": 0.7224, "panda_mae_iqr": 0.2385,
     "chronos_mae": 0.9582, "chronos_mae_iqr": 0.2089,
     "advantage_mae": +0.2358, "wilcoxon_p": 0.001},
    {"label": "Weather_H336", "horizon": 336, "dataset": "Weather",
     "panda_mae": 0.8481, "panda_mae_iqr": 0.2898,
     "chronos_mae": 1.0843, "chronos_mae_iqr": 0.3507,
     "advantage_mae": +0.2362, "wilcoxon_p": 0.000},
]

print(f"Reconstructed {len(exp21_results)} completed rows.")

Reconstructed 11 completed rows.


In [5]:
df_21 = pd.DataFrame(exp21_results)
pd.DataFrame(exp21_results).to_csv("fixed_exp21_results.csv", index=False)
print(f"Exp 2.1 reconstructed: {len(exp21_results)} rows.")
print(df_21[["label","advantage_mae","wilcoxon_p"]].to_string(index=False))

Exp 2.1 reconstructed: 11 rows.
       label  advantage_mae  wilcoxon_p
   ETTh1_H96        -0.0636       0.844
  ETTh1_H192        -0.0360       0.826
  ETTh1_H336         0.0441       0.649
  ETTh1_H720         0.0267       0.774
   ETTh2_H96         0.0758       0.478
  ETTh2_H192        -0.0191       0.147
  ETTh2_H336         0.1845       0.013
  ETTh2_H720        -0.0112       0.392
 Weather_H96         0.1737       0.000
Weather_H192         0.2358       0.001
Weather_H336         0.2362       0.000


In [ ]:
DATA_DIR = "./ts_data"

print("Univariate Ablation — Weather")
print("H96 and H336 only, n_windows=8")
print("Question: does channel attention drive Panda's Weather advantage?")
print("-" * 70)

data_weather    = load_ts(f"{DATA_DIR}/weather.csv")
ablation_results = []

for h in [96, 336]:
    print(f"\n  H={h}:")

    # Comparison A: Multivariate Panda vs Chronos
    res_mc = evaluate(
        data_weather, h, n_windows=8,
        label=f"Weather_multi_vs_chronos_H{h}",
        fn_a=panda_forecast,
        fn_b=chronos_forecast,
        name_a="panda_multi", name_b="chronos",
    )

    # Comparison B: Univariate Panda vs Multivariate Panda
    # This directly tests whether channel attention is doing the work
    res_um = evaluate(
        data_weather, h, n_windows=8,
        label=f"Weather_uni_vs_multi_H{h}",
        fn_a=panda_forecast_univariate,
        fn_b=panda_forecast,
        name_a="panda_uni", name_b="panda_multi",
    )

    for res, cond in [
        (res_mc, "multi_vs_chronos"),
        (res_um, "uni_vs_multi"),
    ]:
        if res:
            res["condition"] = cond
            ablation_results.append(res)

df_abl = pd.DataFrame(ablation_results)
df_abl.to_csv("fixed_ablation_results.csv", index=False)
print("\nSaved fixed_ablation_results.csv")
print()
print("Key: uni_vs_multi advantage")
print("  > 0, p < 0.05 -> univariate worse than multivariate")
print("                   channel attention contributes to Weather win")
print("  ~ 0            -> temporal architecture alone drives the win")
print("  < 0            -> channel attention hurts on Weather")

Univariate Ablation — Weather
H96 and H336 only, n_windows=8
Question: does channel attention drive Panda's Weather advantage?
----------------------------------------------------------------------

  H=96:


We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of l

  Weather_multi_vs_chronos_H96                      H=  96  panda_multi=0.6113[±0.2649]  chronos=0.8132[±0.0746]  Adv=+0.2019  p=0.004 *
  Weather_uni_vs_multi_H96                          H=  96  panda_uni=0.5541[±0.2829]  panda_multi=0.6113[±0.2649]  Adv=+0.0572  p=0.074 ~

  H=336:


We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 


In [3]:
DATA_DIR = "./ts_data"

print("Univariate Ablation — Weather")
print("H96 and H336 only, n_windows=8")
print("Question: does channel attention drive Panda's Weather advantage?")
print("-" * 70)

data_weather    = load_ts(f"{DATA_DIR}/weather.csv")
ablation_results = []

for h in [336]:
    print(f"\n  H={h}:")

    # Comparison A: Multivariate Panda vs Chronos
    res_mc = evaluate(
        data_weather, h, n_windows=8,
        label=f"Weather_multi_vs_chronos_H{h}",
        fn_a=panda_forecast,
        fn_b=chronos_forecast,
        name_a="panda_multi", name_b="chronos",
    )

    # Comparison B: Univariate Panda vs Multivariate Panda
    # This directly tests whether channel attention is doing the work
    res_um = evaluate(
        data_weather, h, n_windows=8,
        label=f"Weather_uni_vs_multi_H{h}",
        fn_a=panda_forecast_univariate,
        fn_b=panda_forecast,
        name_a="panda_uni", name_b="panda_multi",
    )

    for res, cond in [
        (res_mc, "multi_vs_chronos"),
        (res_um, "uni_vs_multi"),
    ]:
        if res:
            res["condition"] = cond
            ablation_results.append(res)

df_abl = pd.DataFrame(ablation_results)
df_abl.to_csv("fixed_ablation_results.csv", index=False)
print("\nSaved fixed_ablation_results.csv")
print()
print("Key: uni_vs_multi advantage")
print("  > 0, p < 0.05 -> univariate worse than multivariate")
print("                   channel attention contributes to Weather win")
print("  ~ 0            -> temporal architecture alone drives the win")
print("  < 0            -> channel attention hurts on Weather")

Univariate Ablation — Weather
H96 and H336 only, n_windows=8
Question: does channel attention drive Panda's Weather advantage?
----------------------------------------------------------------------

  H=336:


We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of l

  Weather_multi_vs_chronos_H336                     H= 336  panda_multi=0.8762[±0.2638]  chronos=0.9987[±0.3059]  Adv=+0.1226  p=0.020 *


ValueError: operands could not be broadcast together with shapes (21,336) (21,128) 

In [5]:
# Rerun only the part that failed
data_weather = load_ts(f"{DATA_DIR}/weather.csv")

res_um_336 = evaluate(
    data_weather, 336, n_windows=8,
    label="Weather_uni_vs_multi_H336",
    fn_a=panda_forecast_univariate,
    fn_b=panda_forecast,
    name_a="panda_uni", name_b="panda_multi",
)

if res_um_336:
    res_um_336["condition"] = "uni_vs_multi"
    print(res_um_336)

  Weather_uni_vs_multi_H336                         H= 336  panda_uni=0.8467[±0.2862]  panda_multi=0.8762[±0.2638]  Adv=+0.0295  p=0.371
{'label': 'Weather_uni_vs_multi_H336', 'horizon': 336, 'name_a': 'panda_uni', 'name_b': 'panda_multi', 'panda_uni_mae': 0.8466947376728058, 'panda_uni_iqr': 0.28624722361564636, 'panda_multi_mae': 0.8761635720729828, 'panda_multi_iqr': 0.2638253718614578, 'advantage_mae': 0.029468834400177002, 'wilcoxon_p': 0.37109375, 'condition': 'uni_vs_multi'}


In [6]:
def simulate_burgers_stable(T=1000, N_x=128, nu=0.005, seed=42):
    rng       = np.random.default_rng(seed)
    dx        = 2 * np.pi / N_x
    dt        = min(0.4 * dx**2 / (2*nu + 1e-10), 0.4*dx, 0.05)
    dt_record = 0.01
    n_sub     = max(1, int(np.ceil(dt_record / dt)))
    dt_act    = dt_record / n_sub

    k       = fftfreq(N_x, d=1.0/N_x).astype(complex)
    dealias = np.abs(k) <= N_x // 3
    L_op    = -nu * k**2

    u0_hat  = np.zeros(N_x, dtype=complex)
    rng2    = np.random.default_rng(seed)
    for m in range(1, 6):
        amp = rng2.standard_normal() + 1j*rng2.standard_normal()
        u0_hat[m]       += amp
        u0_hat[N_x - m] += np.conj(amp)
    u0_hat *= dealias

    def rhs(u_hat):
        u  = np.real(ifft(u_hat))
        nl = fft(0.5 * u**2) * dealias
        return L_op * u_hat - 1j * k * nl

    U     = np.zeros((T, N_x), dtype=np.float32)
    u_hat = u0_hat.copy()
    for t in range(T):
        U[t] = np.real(ifft(u_hat)).astype(np.float32)
        for _ in range(n_sub):
            k1    = rhs(u_hat)
            k2    = rhs(u_hat + 0.5*dt_act*k1)
            k3    = rhs(u_hat + 0.5*dt_act*k2)
            k4    = rhs(u_hat +     dt_act*k3)
            u_hat = u_hat + (dt_act/6)*(k1+2*k2+2*k3+k4)
            u_hat *= dealias
            if not np.isfinite(u_hat).all():
                print(f"    Diverged at t={t}")
                return U[:t]
    return U


def pca_reduction(U, n_components):
    U_c  = U - U.mean(axis=0, keepdims=True)
    n_c  = min(n_components, min(U_c.shape)-1)
    _, _, Vt = svd(U_c, full_matrices=False)
    return (U_c @ Vt[:n_c].T).astype(np.float32)


# Test solver
print("Testing Burgers solver:")
for nu_t in [2.0, 1.0, 0.1, 0.005]:
    U = simulate_burgers_stable(T=50, nu=nu_t)
    ok = "OK" if len(U)==50 else f"FAILED at step {len(U)}"
    print(f"  nu={nu_t:.3f}: range=[{U.min():.3f},{U.max():.3f}]  {ok}")

Testing Burgers solver:
  nu=2.000: range=[-0.060,0.063]  OK
  nu=1.000: range=[-0.060,0.063]  OK
  nu=0.100: range=[-0.060,0.063]  OK
  nu=0.005: range=[-0.060,0.063]  OK


In [7]:
def fft_decompose_channel(series_1d, period):
    """
    Remove dominant periodic components via FFT.
    Context window only — no future information used.
    """
    N     = len(series_1d)
    X     = np.fft.rfft(series_1d)
    freqs = np.fft.rfftfreq(N)

    det_mask    = np.zeros(len(X), dtype=bool)
    det_mask[0] = True
    n_trend     = max(1, int(0.01 * len(X)))
    det_mask[:n_trend] = True

    fund_freq = 1.0 / period
    for m in range(1, int(N / period) + 1):
        idx = np.argmin(np.abs(freqs - m * fund_freq))
        if idx < len(X):
            det_mask[max(0, idx-1):idx+2] = True

    X_det         = np.zeros_like(X)
    X_det[det_mask] = X[det_mask]
    deterministic = np.fft.irfft(X_det, n=N)
    return deterministic, series_1d - deterministic


def project_forward(det_ctx, period, horizon):
    """
    Project deterministic component forward from context.
    Linear trend extrapolation + repeated seasonal cycle.
    """
    N     = len(det_ctx)
    X     = np.fft.rfft(det_ctx)
    n_t   = max(1, int(0.01 * len(X)))
    X_tr  = np.zeros_like(X)
    X_tr[:n_t] = X[:n_t]
    trend_ctx  = np.fft.irfft(X_tr, n=N)
    seas_ctx   = det_ctx - trend_ctx

    coeffs     = np.polyfit(np.arange(N), trend_ctx, 1)
    trend_proj = np.polyval(coeffs, np.arange(N, N+horizon))

    template   = seas_ctx[-period:]
    seas_proj  = np.tile(template, int(np.ceil(horizon/period)))[:horizon]

    return trend_proj + seas_proj


def evaluate_decomposed_fft(data_CT, horizon, period=24,
                             n_windows=8, label=""):
    """
    Runs vanilla and FFT-decomposed evaluation in one pass.
    data_CT: (C, T) raw.
    Returns (vanilla_summary, decomposed_summary).
    """
    C, T      = data_CT.shape
    max_start = T - CONTEXT_LEN - horizon
    if max_start <= 0:
        print(f"  [SKIP] {label} H={horizon}")
        return None, None

    starts = np.linspace(0, max_start, n_windows, dtype=int)
    mae_pv, mae_cv = [], []   # vanilla
    mae_pd, mae_cd = [], []   # decomposed

    for s in starts:
        ctx_raw          = data_CT[:, s : s + CONTEXT_LEN]
        tgt_raw          = data_CT[:, s + CONTEXT_LEN
                                      : s + CONTEXT_LEN + horizon]
        ctx_norm, mu, std = instance_norm_window(ctx_raw)
        tgt_norm          = (tgt_raw - mu) / std

        # Vanilla
        mae_pv.append(mae(tgt_norm, panda_forecast(ctx_norm, horizon)))
        mae_cv.append(mae(tgt_norm, chronos_forecast(ctx_norm, horizon)))

        # FFT decomposition on raw context
        ctx_res  = np.zeros_like(ctx_raw)
        det_proj = np.zeros((C, horizon))
        for c in range(C):
            det_c, res_c    = fft_decompose_channel(ctx_raw[c], period)
            ctx_res[c]      = res_c
            det_proj[c]     = project_forward(det_c, period, horizon)

        ctx_res_norm, mu_r, std_r = instance_norm_window(ctx_res)
        p_r = panda_forecast(ctx_res_norm, horizon)
        c_r = chronos_forecast(ctx_res_norm, horizon)

        # Reconstruct to original scale, then normalise with vanilla stats
        p_full = ((p_r * std_r + mu_r + det_proj) - mu) / std
        c_full = ((c_r * std_r + mu_r + det_proj) - mu) / std

        mae_pd.append(mae(tgt_norm, p_full))
        mae_cd.append(mae(tgt_norm, c_full))

    def _summary(mae_a, mae_b, tag, cond):
        diff = np.array(mae_b) - np.array(mae_a)
        try:
            _, pval = wilcoxon(diff, alternative="greater") \
                if np.any(diff != 0) else (0, 1.0)
        except Exception:
            pval = np.nan
        adv = np.median(mae_b) - np.median(mae_a)
        sig = " *" if pval < 0.05 else (" ~" if pval < 0.10 else "")
        iqr_a = np.percentile(mae_a,75)-np.percentile(mae_a,25)
        iqr_b = np.percentile(mae_b,75)-np.percentile(mae_b,25)
        print(
            f"  {tag:50s}  H={horizon:4d}  "
            f"P={np.median(mae_a):.4f}[±{iqr_a:.4f}]  "
            f"C={np.median(mae_b):.4f}[±{iqr_b:.4f}]  "
            f"Adv={adv:+.4f}  p={pval:.3f}{sig}"
        )
        return {
            "label":tag, "horizon":horizon, "dataset":label,
            "condition":cond,
            "panda_mae":np.median(mae_a), "panda_iqr":iqr_a,
            "chronos_mae":np.median(mae_b), "chronos_iqr":iqr_b,
            "advantage_mae":adv, "wilcoxon_p":pval,
        }

    r_v = _summary(mae_pv, mae_cv, f"{label}_vanilla_H{horizon}",   "vanilla")
    r_d = _summary(mae_pd, mae_cd, f"{label}_decomp_H{horizon}",    "decomp_fft")
    return r_v, r_d


print("Exp 2.2: FFT Decomposition (no oracle leakage)")
print("n_windows=8, H=[96, 336]")
print("-" * 70)

dataset_periods = {"ETTh1":24, "ETTh2":24, "Weather":144}
exp22_results   = []

for dname, dpath in {
    "ETTh1"  : f"{DATA_DIR}/ETTh1.csv",
    "ETTh2"  : f"{DATA_DIR}/ETTh2.csv",
    "Weather": f"{DATA_DIR}/weather.csv",
}.items():
    data   = load_ts(dpath)
    period = dataset_periods[dname]
    print(f"\n  {dname}: shape {data.shape}, period={period}")
    for h in [96, 336]:
        rv, rd = evaluate_decomposed_fft(
            data, h, period=period, n_windows=8, label=dname
        )
        if rv: exp22_results.append(rv)
        if rd: exp22_results.append(rd)

df_22 = pd.DataFrame(exp22_results)
df_22.to_csv("fixed_exp22_results.csv", index=False)
print("\nSaved fixed_exp22_results.csv")

Exp 2.2: FFT Decomposition (no oracle leakage)
n_windows=8, H=[96, 336]
----------------------------------------------------------------------

  ETTh1: shape (7, 17420), period=24


We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of l

  ETTh1_vanilla_H96                                   H=  96  P=0.7205[±0.1949]  C=0.8157[±0.2682]  Adv=+0.0952  p=0.422
  ETTh1_decomp_H96                                    H=  96  P=0.8032[±0.2060]  C=0.8293[±0.1292]  Adv=+0.0261  p=0.578


We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of l

  ETTh1_vanilla_H336                                  H= 336  P=0.8407[±0.4892]  C=0.8856[±0.4932]  Adv=+0.0449  p=0.273
  ETTh1_decomp_H336                                   H= 336  P=1.0474[±0.5172]  C=1.0174[±0.5246]  Adv=-0.0300  p=0.770

  ETTh2: shape (7, 17420), period=24


...

  ETTh2_vanilla_H336                                  H= 336  P=1.2596[±0.5309]  C=1.1578[±0.7595]  Adv=-0.1018  p=0.371
  ETTh2_decomp_H336                                   H= 336  P=1.3294[±0.7693]  C=1.2299[±0.7454]  Adv=-0.0995  p=0.945

  Weather: shape (21, 52696), period=144


We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of l

  Weather_vanilla_H96                                 H=  96  P=0.6082[±0.2633]  C=0.6880[±0.1875]  Adv=+0.0798  p=0.020 *
  Weather_decomp_H96                                  H=  96  P=0.6771[±0.2792]  C=0.6871[±0.2295]  Adv=+0.0100  p=0.230


We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of l

  Weather_vanilla_H336                                H= 336  P=0.8762[±0.2638]  C=0.9897[±0.2127]  Adv=+0.1136  p=0.004 *
  Weather_decomp_H336                                 H= 336  P=1.0609[±0.1046]  C=1.0760[±0.1152]  Adv=+0.0151  p=0.020 *

Saved fixed_exp22_results.csv


In [8]:
def uniform_subsample(U, n_points):
    idx = np.linspace(0, U.shape[1]-1, n_points, dtype=int)
    return U[:, idx].astype(np.float32)


def variance_stratified_subsample(U, n_points, pct=10):
    """Uniform spacing but excludes bottom pct% variance locations."""
    variances = U.var(axis=0)
    threshold = np.percentile(variances, pct)
    valid     = np.where(variances >= threshold)[0]
    if len(valid) < n_points:
        valid = np.arange(U.shape[1])
    selected  = valid[np.linspace(0, len(valid)-1, n_points, dtype=int)]
    return U[:, selected].astype(np.float32)


def compute_dynamical_features(U):
    from scipy.signal import periodogram
    feats = []
    for x in range(U.shape[1]):
        ts      = U[:, x].astype(float)
        _, pwr  = periodogram(ts)
        p       = pwr / (pwr.sum() + 1e-10)
        p       = p[p > 1e-10]
        feats.append([
            float(np.std(ts)),
            float(np.mean(np.abs(ts))),
            float(np.percentile(np.abs(ts), 90)),
            float(-np.sum(p * np.log(p))),
        ])
    F = np.array(feats)
    return (F - F.min(0)) / (F.max(0) - F.min(0) + 1e-8)


def farthest_point_sampling(features, n_points, seed=42):
    rng  = np.random.default_rng(seed)
    N    = features.shape[0]
    sel  = [int(rng.integers(0, N))]
    dist = np.full(N, np.inf)
    for _ in range(n_points - 1):
        d    = np.linalg.norm(features - features[sel[-1]], axis=1)
        dist = np.minimum(dist, d)
        dc   = dist.copy()
        dc[sel] = -np.inf
        sel.append(int(np.argmax(dc)))
    return sorted(sel)


def diversity_subsample(U, n_points, seed=42):
    F   = compute_dynamical_features(U)
    idx = farthest_point_sampling(F, n_points, seed=seed)
    return U[:, idx].astype(np.float32)


print("Subsampling Comparison — Burgers nu=0.005 and nu=0.05")
print("Methods: Uniform, Stratified_Uniform, PCA, Diversity")
print("n_windows=8")
print("-" * 70)

N_CH        = 16
sub_results = []

for nu in [0.05, 0.005]:
    print(f"\n  nu={nu}:")
    U = simulate_burgers_stable(T=1000, nu=nu)
    if len(U) < CONTEXT_LEN + 128 + 10:
        print(f"    Too short, skipping")
        continue

    for method_name, U_ch in [
        ("Uniform",            uniform_subsample(U, N_CH)),
        ("Stratified_Uniform", variance_stratified_subsample(U, N_CH)),
        ("PCA",                pca_reduction(U, N_CH)),
        ("Diversity",          diversity_subsample(U, N_CH)),
    ]:
        data = U_ch.T   # (N_CH, T) raw
        res  = evaluate(data, 128, n_windows=8,
                        label=f"Burgers_nu={nu}_{method_name}")
        if res:
            res["nu"]     = nu
            res["method"] = method_name
            sub_results.append(res)

df_sub = pd.DataFrame(sub_results)
df_sub.to_csv("fixed_subsampling_results.csv", index=False)
print("\nSaved fixed_subsampling_results.csv")
print()
print("Key question: does Diversity beat Stratified_Uniform?")
print("  If yes: diversity benefit is real, not just avoiding nodal points")
print("  If no:  previous Exp 1.1 result was an artifact")

Subsampling Comparison — Burgers nu=0.005 and nu=0.05
Methods: Uniform, Stratified_Uniform, PCA, Diversity
n_windows=8
----------------------------------------------------------------------

  nu=0.05:


We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of l

  Burgers_nu=0.05_Uniform                           H= 128  panda=0.0317[±0.0091]  chronos=0.0765[±0.0295]  Adv=+0.0448  p=0.004 *


We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of l

  Burgers_nu=0.05_Stratified_Uniform                H= 128  panda=0.0296[±0.0054]  chronos=0.0736[±0.0244]  Adv=+0.0441  p=0.004 *


...

  Burgers_nu=0.005_Stratified_Uniform               H= 128  panda=0.0301[±0.0013]  chronos=0.1779[±0.0780]  Adv=+0.1478  p=0.004 *


We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of l

  Burgers_nu=0.005_PCA                              H= 128  panda=0.1509[±0.0840]  chronos=0.3029[±0.2280]  Adv=+0.1521  p=0.004 *


We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of l

  Burgers_nu=0.005_Diversity                        H= 128  panda=0.0300[±0.0046]  chronos=0.2475[±0.1989]  Adv=+0.2175  p=0.004 *

Saved fixed_subsampling_results.csv

Key question: does Diversity beat Stratified_Uniform?
  If yes: diversity benefit is real, not just avoiding nodal points
  If no:  previous Exp 1.1 result was an artifact


In [9]:
print("=" * 70)
print("ALL FIXED EXPERIMENT RESULTS")
print("=" * 70)

for name, df in [
    ("Exp 2.1 (reconstructed)",   df_21),
    ("Exp 2.2 FFT decomposition", df_22),
    ("Burgers sweep",             df_burgers),
    ("Univariate ablation",       df_abl),
    ("Subsampling",               df_sub),
]:
    print(f"\n--- {name} ---")
    cols = [c for c in [
        "label","horizon","condition","method","nu",
        "advantage_mae","wilcoxon_p"
    ] if c in df.columns]
    print(df[cols].round(4).to_string(index=False))

ALL FIXED EXPERIMENT RESULTS


NameError: name 'df_21' is not defined

In [10]:
print("\nBurgers Viscosity Sweep")
print("nu >= 0.5: non-chaotic baseline")
print("nu < 0.1:  chaotic regime")
print("-" * 70)

# non-chaotic -> transition -> chaotic
nu_values       = [2.0, 1.0, 0.5, 0.1, 0.05, 0.02, 0.01, 0.005]
N_COMP          = 16
burgers_results = []

for nu in nu_values:
    U = simulate_burgers_stable(T=1000, nu=nu)
    if len(U) < CONTEXT_LEN + 128 + 10:
        print(f"  nu={nu}: too short ({len(U)} steps), skipping")
        continue

    data = pca_reduction(U, N_COMP).T   # (N_COMP, T) raw
    res  = evaluate(data, 128, n_windows=8,
                    label=f"Burgers_PCA_nu={nu:.3f}")
    if res:
        res["nu"] = nu
        burgers_results.append(res)

df_burgers = pd.DataFrame(burgers_results)
df_burgers.to_csv("fixed_burgers_results.csv", index=False)
print("\nSaved fixed_burgers_results.csv")
print()
print("Key question: does advantage flip sign at non-chaotic nu values?")
print("If yes: supports chaos-threshold hypothesis for PDEs")
print("If no:  advantage may reflect signal variance, not chaos")


Burgers Viscosity Sweep
nu >= 0.5: non-chaotic baseline
nu < 0.1:  chaotic regime
----------------------------------------------------------------------


We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of l

  Burgers_PCA_nu=2.000                              H= 128  panda=0.0152[±0.0034]  chronos=0.0191[±0.0187]  Adv=+0.0039  p=0.273


We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of l

  Burgers_PCA_nu=1.000                              H= 128  panda=0.0186[±0.0068]  chronos=0.0568[±0.0867]  Adv=+0.0382  p=0.004 *


...

  Burgers_PCA_nu=0.020                              H= 128  panda=0.1394[±0.0575]  chronos=0.2593[±0.1432]  Adv=+0.1199  p=0.004 *


We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of l

  Burgers_PCA_nu=0.010                              H= 128  panda=0.1375[±0.0745]  chronos=0.2325[±0.0680]  Adv=+0.0950  p=0.004 *


We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of l

  Burgers_PCA_nu=0.005                              H= 128  panda=0.1509[±0.0840]  chronos=0.2732[±0.0448]  Adv=+0.1223  p=0.004 *

Saved fixed_burgers_results.csv

Key question: does advantage flip sign at non-chaotic nu values?
If yes: supports chaos-threshold hypothesis for PDEs
If no:  advantage may reflect signal variance, not chaos


In [ ]:
def project_deterministic_forward(trend_ctx, seasonal_ctx,
                                   horizon, period):
    """
    Project trend and seasonal forward using only context information.
    No future data used.
    trend_ctx:    (T_ctx,)
    seasonal_ctx: (T_ctx,)
    Returns: (horizon,)
    """
    # Linear extrapolation of trend
    t_in         = np.arange(len(trend_ctx))
    coeffs       = np.polyfit(t_in, trend_ctx, 1)
    t_out        = np.arange(len(trend_ctx), len(trend_ctx) + horizon)
    trend_proj   = np.polyval(coeffs, t_out)

    # Repeat last full seasonal cycle
    template     = seasonal_ctx[-period:]
    n_repeats    = int(np.ceil(horizon / period))
    seasonal_proj = np.tile(template, n_repeats)[:horizon]

    return trend_proj + seasonal_proj


def evaluate_decomposed_fixed(data_CT, horizon, period=24,
                               n_windows=20, label=""):
    """
    STL on context window only.
    Deterministic projected forward naively.
    No oracle leakage.
    data_CT: (C, T) RAW.
    """
    C, T      = data_CT.shape
    max_start = T - CONTEXT_LEN - horizon
    if max_start <= 0:
        print(f"  [SKIP] {label}: not enough data")
        return None

    starts = np.linspace(0, max_start, n_windows, dtype=int)

    mae_panda, mae_chronos   = [], []
    mae_panda_vanilla, mae_chronos_vanilla = [], []

    for s in starts:
        ctx_raw  = data_CT[:, s : s + CONTEXT_LEN]
        tgt_raw  = data_CT[:, s + CONTEXT_LEN : s + CONTEXT_LEN + horizon]

        # --- Vanilla (no decomposition) ---
        ctx_norm, mu, std = instance_norm_window(ctx_raw)
        tgt_norm = (tgt_raw - mu) / std
        mae_panda_vanilla.append(
            mae(tgt_norm, panda_forecast(ctx_norm, horizon))
        )
        mae_chronos_vanilla.append(
            mae(tgt_norm, chronos_forecast(ctx_norm, horizon))
        )

        # --- Decomposition ---
        ctx_residual = np.zeros_like(ctx_raw)   # (C, CONTEXT_LEN)
        det_proj     = np.zeros((C, horizon))

        for c in range(C):
            try:
                stl        = STL(ctx_raw[c], period=period, robust=True)
                result     = stl.fit()
                ctx_residual[c] = result.resid
                det_proj[c]     = project_deterministic_forward(
                    result.trend, result.seasonal, horizon, period
                )
            except Exception:
                ctx_residual[c] = ctx_raw[c]
                det_proj[c]     = 0.0

        # Normalise residual context
        ctx_res_norm, mu_r, std_r = instance_norm_window(ctx_residual)
        # Note: target normalised using vanilla stats for fair comparison
        tgt_norm_for_eval = (tgt_raw - mu) / std

        p_res = panda_forecast(ctx_res_norm, horizon)
        c_res = chronos_forecast(ctx_res_norm, horizon)

        # Reconstruct: denormalise residual forecast, add deterministic
        p_full_raw = p_res * std_r + mu_r + det_proj
        c_full_raw = c_res * std_r + mu_r + det_proj

        # Re-normalise for metric computation
        p_full_norm = (p_full_raw - mu) / std
        c_full_norm = (c_full_raw - mu) / std

        mae_panda.append(mae(tgt_norm_for_eval, p_full_norm))
        mae_chronos.append(mae(tgt_norm_for_eval, c_full_norm))

    def _summarise(vals_a, vals_b, tag):
        diff = np.array(vals_b) - np.array(vals_a)
        try:
            _, pval = wilcoxon(diff, alternative="greater") \
                if np.any(diff != 0) else (0, 1.0)
        except Exception:
            pval = np.nan
        adv = np.median(vals_b) - np.median(vals_a)
        sig = " *" if pval < 0.05 else (" ~" if pval < 0.10 else "")
        print(
            f"  {tag:50s}  H={horizon:4d}  "
            f"Panda={np.median(vals_a):.4f}"
            f"[±{np.percentile(vals_a,75)-np.percentile(vals_a,25):.4f}]  "
            f"Chronos={np.median(vals_b):.4f}"
            f"[±{np.percentile(vals_b,75)-np.percentile(vals_b,25):.4f}]  "
            f"Adv={adv:+.4f}  p={pval:.3f}{sig}"
        )
        return {
            "label": tag, "horizon": horizon, "dataset": label,
            "panda_mae": np.median(vals_a),
            "panda_mae_iqr": np.percentile(vals_a,75)-np.percentile(vals_a,25),
            "chronos_mae": np.median(vals_b),
            "chronos_mae_iqr": np.percentile(vals_b,75)-np.percentile(vals_b,25),
            "advantage_mae": adv, "wilcoxon_p": pval,
        }

    r_van  = _summarise(mae_panda_vanilla, mae_chronos_vanilla,
                        f"{label}_vanilla")
    r_decp = _summarise(mae_panda, mae_chronos,
                        f"{label}_decomp_fixed")
    return r_van, r_decp


print("Fixed Exp 2.2: Decomposition (no oracle leakage)")
print("n_windows=20, context-only STL, naive projection")
print("-" * 75)

exp22_results = []

for dname, dpath in datasets.items():
    data   = load_ts(dpath)
    period = dataset_periods[dname]
    print(f"\n  {dname}: shape {data.shape}, period={period}")

    for h in [96, 192, 336, 720]:
        r_van, r_decp = evaluate_decomposed_fixed(
            data, h, period=period,
            n_windows=20, label=dname
        )
        if r_van:
            r_van["condition"]  = "vanilla"
            exp22_results.append(r_van)
        if r_decp:
            r_decp["condition"] = "decomp_fixed"
            exp22_results.append(r_decp)

df_22 = pd.DataFrame(exp22_results)
df_22.to_csv("fixed_exp22_results.csv", index=False)
print("\nSaved fixed_exp22_results.csv")

In [ ]:
def simulate_burgers_stable(T=1000, N_x=128, nu=0.005, seed=42):
    """
    Burgers with adaptive dt — stable at any nu including nu=1.0, 2.0.
    """
    rng = np.random.default_rng(seed)
    dx  = 2 * np.pi / N_x

    # Stability conditions
    dt_diff = 0.4 * dx**2 / (2 * nu + 1e-10)
    dt_adv  = 0.4 * dx
    dt      = min(dt_diff, dt_adv, 0.05)

    # Recording interval always 0.01 regardless of nu
    dt_record = 0.01
    n_substeps = max(1, int(np.ceil(dt_record / dt)))
    dt_actual  = dt_record / n_substeps

    k       = fftfreq(N_x, d=1.0/N_x).astype(complex)
    k_max   = N_x // 3
    dealias = np.abs(k) <= k_max
    L_op    = -nu * k**2

    u0_hat = np.zeros(N_x, dtype=complex)
    for m in range(1, 6):
        amp = rng.standard_normal() + 1j * rng.standard_normal()
        u0_hat[m]       += amp
        u0_hat[N_x - m] += np.conj(amp)
    u0_hat *= dealias

    def rhs_hat(u_hat):
        u_phys = np.real(ifft(u_hat))
        nonlin = fft(0.5 * u_phys**2) * dealias
        return L_op * u_hat - 1j * k * nonlin

    U     = np.zeros((T, N_x), dtype=np.float32)
    u_hat = u0_hat.copy()

    for t in range(T):
        U[t] = np.real(ifft(u_hat)).astype(np.float32)
        for _ in range(n_substeps):
            k1    = rhs_hat(u_hat)
            k2    = rhs_hat(u_hat + 0.5*dt_actual*k1)
            k3    = rhs_hat(u_hat + 0.5*dt_actual*k2)
            k4    = rhs_hat(u_hat +     dt_actual*k3)
            u_hat = u_hat + (dt_actual/6.0)*(k1+2*k2+2*k3+k4)
            u_hat *= dealias
            if not np.isfinite(u_hat).all():
                print(f"    Diverged at t={t}")
                return U[:t]
    return U


def pca_reduction(U, n_components):
    U_c  = U - U.mean(axis=0, keepdims=True)
    n_c  = min(n_components, min(U_c.shape)-1)
    _, _, Vt = svd(U_c, full_matrices=False)
    return (U_c @ Vt[:n_c].T).astype(np.float32)


# Test solver first
print("Testing stable Burgers solver:")
for nu_test in [2.0, 1.0, 0.1, 0.01, 0.005]:
    U = simulate_burgers_stable(T=50, nu=nu_test)
    status = "OK" if len(U) == 50 else f"FAILED at step {len(U)}"
    print(f"  nu={nu_test:.3f}: range=[{U.min():.3f},{U.max():.3f}]  {status}")

In [ ]:
print("Fixed Exp: Univariate Ablation on Weather")
print("Question: is channel attention driving Panda's Weather advantage?")
print("-" * 75)

data_weather = load_ts(f"{DATA_DIR}/weather.csv")
ablation_results = []

for h in [96, 192, 336, 720]:
    # Multivariate Panda vs Chronos (standard comparison)
    res_multi = evaluate(
        data_weather, h, n_windows=20,
        label=f"Weather_multivariate_H{h}",
        forecast_fn_a=lambda ctx, hh: panda_forecast(ctx, hh),
        forecast_fn_b=lambda ctx, hh: chronos_forecast(ctx, hh),
        name_a="panda_multi", name_b="chronos",
    )

    # Univariate Panda vs Multivariate Panda
    res_uni = evaluate(
        data_weather, h, n_windows=20,
        label=f"Weather_univariate_H{h}",
        forecast_fn_a=lambda ctx, hh: panda_forecast_univariate(ctx, hh),
        forecast_fn_b=lambda ctx, hh: panda_forecast(ctx, hh),
        name_a="panda_uni", name_b="panda_multi",
    )

    if res_multi:
        res_multi["condition"] = "multi_vs_chronos"
        res_multi["horizon"]   = h
        ablation_results.append(res_multi)
    if res_uni:
        res_uni["condition"] = "uni_vs_multi"
        res_uni["horizon"]   = h
        ablation_results.append(res_uni)

df_abl = pd.DataFrame(ablation_results)
df_abl.to_csv("fixed_ablation_results.csv", index=False)

print("\nInterpretation:")
print("  uni_vs_multi advantage > 0, p < 0.05:")
print("    -> univariate Panda WORSE than multivariate")
print("    -> channel attention helps on Weather")
print("  uni_vs_multi advantage ~ 0:")
print("    -> temporal architecture drives the advantage, not coupling")
print("  uni_vs_multi advantage < 0:")
print("    -> channel attention hurts on Weather (interesting failure mode)")

In [ ]:
def uniform_subsample(U, n_points):
    idx = np.linspace(0, U.shape[1]-1, n_points, dtype=int)
    return U[:, idx].astype(np.float32)


def variance_stratified_subsample(U, n_points,
                                   var_threshold_percentile=10):
    """
    Uniform spacing but excludes near-zero-variance locations.
    Fair comparison for diversity subsampling.
    """
    variances = U.var(axis=0)
    threshold = np.percentile(variances, var_threshold_percentile)
    valid_idx = np.where(variances >= threshold)[0]

    if len(valid_idx) < n_points:
        valid_idx = np.arange(U.shape[1])

    selected = valid_idx[
        np.linspace(0, len(valid_idx)-1, n_points, dtype=int)
    ]
    return U[:, selected].astype(np.float32), selected


def compute_dynamical_features(U):
    from scipy.signal import periodogram
    T, N_x = U.shape
    features = []
    for x in range(N_x):
        ts = U[:, x].astype(float)
        freqs, power = periodogram(ts)
        p_norm = power / (power.sum() + 1e-10)
        p_norm = p_norm[p_norm > 1e-10]
        spec_entropy = float(-np.sum(p_norm * np.log(p_norm)))
        dom_freq = float(freqs[np.argmax(power[1:])+1])
        features.append([
            float(np.mean(ts)),
            float(np.std(ts)),
            float(np.mean(np.abs(ts))),
            float(np.percentile(np.abs(ts), 90)),
            dom_freq,
            spec_entropy,
        ])
    F = np.array(features)
    F_min = F.min(axis=0, keepdims=True)
    F_max = F.max(axis=0, keepdims=True)
    return (F - F_min) / (F_max - F_min + 1e-8)


def farthest_point_sampling(features, n_points, seed=42):
    rng      = np.random.default_rng(seed)
    N_x      = features.shape[0]
    selected = [int(rng.integers(0, N_x))]
    dists    = np.full(N_x, np.inf)

    for _ in range(n_points - 1):
        last    = selected[-1]
        d       = np.linalg.norm(features - features[last], axis=1)
        dists   = np.minimum(dists, d)
        dists_copy = dists.copy()
        dists_copy[selected] = -np.inf
        selected.append(int(np.argmax(dists_copy)))

    return sorted(selected)


def diversity_subsample(U, n_points, seed=42):
    features = compute_dynamical_features(U)
    indices  = farthest_point_sampling(features, n_points, seed=seed)
    return U[:, indices].astype(np.float32), indices


print("Fixed Exp 1.1: Subsampling comparison")
print("Adds variance-stratified uniform as a fair control")
print("-" * 75)

N_CHANNELS     = 16
nu_values_sub  = [0.05, 0.01, 0.005]
subsampling_results = []

for nu in nu_values_sub:
    print(f"\n  nu={nu}:")
    U = simulate_burgers_stable(T=1000, nu=nu)
    if len(U) < CONTEXT_LEN + 128 + 10:
        print(f"    Too short, skipping")
        continue

    methods = {}

    # Method 1: Uniform
    methods["Uniform"] = uniform_subsample(U, N_CHANNELS)

    # Method 2: Variance-stratified uniform (fair control)
    U_strat, _ = variance_stratified_subsample(U, N_CHANNELS)
    methods["Stratified_Uniform"] = U_strat

    # Method 3: PCA
    methods["PCA"] = pca_reduction(U, N_CHANNELS)

    # Method 4: Diversity
    U_div, _ = diversity_subsample(U, N_CHANNELS)
    methods["Diversity"] = U_div

    for method_name, U_ch in methods.items():
        data = U_ch.T  # (N_COMP, T) — raw
        res  = evaluate(data, 128, n_windows=20,
                        label=f"Burgers_nu={nu}_{method_name}")
        if res:
            res["nu"]     = nu
            res["method"] = method_name
            subsampling_results.append(res)

df_sub = pd.DataFrame(subsampling_results)
df_sub.to_csv("fixed_subsampling_results.csv", index=False)
print("\nSaved fixed_subsampling_results.csv")

In [ ]:
print("\n" + "="*75)
print("FIXED EXPERIMENTS SUMMARY")
print("="*75)

for name, df in [
    ("Exp 2.1 Standard Horizons", df_21),
    ("Exp 2.2 Decomposition",     df_22),
    ("Burgers Sweep",             df_burgers),
    ("Univariate Ablation",       df_abl),
    ("Subsampling Comparison",    df_sub),
]:
    print(f"\n--- {name} ---")
    cols = ["label","horizon","advantage_mae","wilcoxon_p"] \
           if "label" in df.columns else list(df.columns[:6])
    cols = [c for c in cols if c in df.columns]
    print(df[cols].round(4).to_string(index=False))